<a href="https://colab.research.google.com/github/Kamalll7/Final_Project_PCD-Kelompok_6/blob/main/notebook/01_Helmet_Detection_CHT_HOG_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## 1. Pendahuluan

### 1.1 Latar Belakang Singkat

Sepeda motor adalah kendaraan yang paling banyak digunakan di Indonesia. Undang-Undang
Nomor 22 Tahun 2009 mewajibkan pengendara sepeda motor memakai helm yang memenuhi standar.
Pengawasan manual terhadap kepatuhan ini tidak efisien pada skala lalu lintas nyata, sehingga
diperlukan sistem deteksi otomatis berbasis pengolahan citra digital.

### 1.2 Rumusan Masalah yang Dijawab Notebook Ini

> **Bagaimana mendeteksi keberadaan helm pada pengendara sepeda motor menggunakan metode
> pengolahan citra digital dan machine learning?** (Rumusan Masalah #1)

### 1.3 Tujuan

Mengembangkan dan mengevaluasi metode deteksi keberadaan helm pada pengendara sepeda motor
menggunakan **CHT** untuk melokalisasi area kepala, **HOG** untuk mengekstraksi deskriptor
tekstur/gradien, dan **SVM** untuk klasifikasi biner.

### 1.4 Mengapa CHT + HOG + SVM?

Berdasarkan kajian pustaka pada proposal (§2.3–§2.5):

- **CHT** efektif melokalisasi objek berbentuk lingkaran. Helm dan kepala manusia sama-sama
  menyerupai lingkaran, sehingga CHT dapat mempersempit area pencarian.
- **CHT saja tidak cukup** — helm dan kepala tanpa helm sama-sama berbentuk lingkaran. Informasi
  geometris semata tidak dapat membedakan keduanya (temuan e Silva et al., 2017).
- **HOG** menangkap perbedaan karakteristik permukaan: helm umumnya halus dan mengilap,
  sedangkan rambut/kepala memiliki tekstur yang lebih kompleks.
- **SVM kernel linear** memberikan keseimbangan terbaik antara akurasi dan kecepatan komputasi
  pada fitur HOG berdimensi tinggi (Hsu et al., 2015).

Kombinasi CHT (lokalisasi) + HOG (deskripsi) inilah yang disebut **deskriptor hybrid** pada proposal.

---
## 2. Setup Environment

Notebook ini dirancang untuk berjalan di **Google Colab** (sesuai §3.10 proposal) maupun secara lokal.

**Perangkat lunak yang digunakan** (§3.10):
`Python`, `OpenCV` (pengolahan citra & CHT), `scikit-image` (HOG), `scikit-learn` (SVM & metrik),
`NumPy`, `Matplotlib`, `Pandas`.

In [ ]:
import sys, subprocess

REQUIRED = ["opencv-python-headless", "scikit-image", "scikit-learn",
            "matplotlib", "pandas", "numpy", "pillow"]

def _ensure(pkgs):
    missing = []
    for p in pkgs:
        mod = {"opencv-python-headless": "cv2", "scikit-image": "skimage",
               "scikit-learn": "sklearn", "pillow": "PIL"}.get(p, p)
        try:
            __import__(mod)
        except ImportError:
            missing.append(p)
    if missing:
        print("Memasang:", missing)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

_ensure(REQUIRED)

In [ ]:
import os, json, random, warnings, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle

from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, cohen_kappa_score,
                             roc_curve, auc, classification_report)

warnings.filterwarnings("ignore")

# ---- Reproducibility -------------------------------------------------------
SEED = 7
random.seed(SEED)
np.random.seed(SEED)

# ---- Tampilan plot ---------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 110,
    "font.size": 9,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
})

print("NumPy       :", np.__version__)
print("OpenCV      :", cv2.__version__)
import sklearn, skimage
print("scikit-learn:", sklearn.__version__)
print("scikit-image:", skimage.__version__)

---
## 3. Akuisisi Dataset

Dataset: **Rider, With Helmet, Without Helmet, Number Plate** (Kaggle, `aneesarom`).

Struktur dataset mengikuti format anotasi **YOLO**:

```
dataset/
├── train/
│   ├── images/   *.jpg
│   └── labels/   *.txt      # satu baris per objek: <class> <x_center> <y_center> <w> <h>
└── val/
    ├── images/
    └── labels/
```

Nilai koordinat pada label YOLO **dinormalisasi** terhadap lebar dan tinggi citra (rentang 0–1),
sehingga perlu dikonversi ke koordinat piksel sebelum digunakan.

**Kelas** (`classes.txt`):

| ID | Nama kelas |
|---|---|
| 0 | `with helmet` |
| 1 | `without helmet` |
| 2 | `rider` |
| 3 | `number plate` |

> **Catatan penting.** Dataset ini menyediakan anotasi `with helmet` / `without helmet` yang
> sudah terasosiasi dengan setiap pengendara. Anotasi inilah yang menjadi **ground truth**
> untuk melatih dan mengevaluasi classifier SVM pada jalur helm.

In [ ]:
DATASET_SLUG = "aneesarom/rider-with-helmet-without-helmet-number-plate"
CLASS_NAMES  = ["with helmet", "without helmet", "rider", "number plate"]

def _looks_like_dataset(p: Path) -> bool:
    return (p / "train" / "images").is_dir() and (p / "train" / "labels").is_dir()

def resolve_dataset() -> Path:
    """Cari folder dataset: env var -> path lokal umum -> cache kagglehub -> unduh."""
    candidates = []
    env = os.environ.get("PCD_DATASET_DIR")
    if env:
        candidates.append(Path(env))
    candidates += [Path("./dataset"), Path("../dataset"), Path("/content/dataset"),
                   Path.home() / ".cache/kagglehub/datasets/aneesarom/rider-with-helmet-without-helmet-number-plate"]

    for base in candidates:
        if not base.exists():
            continue
        if _looks_like_dataset(base):
            return base
        # kagglehub menyimpan pada subfolder bernomor versi
        for sub in base.rglob("train"):
            if (sub / "images").is_dir() and (sub / "labels").is_dir():
                return sub.parent

    print("Dataset belum ditemukan secara lokal, mengunduh via kagglehub ...")
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=False)
        import kagglehub
    root = Path(kagglehub.dataset_download(DATASET_SLUG))
    for sub in [root] + list(root.rglob("*")):
        if _looks_like_dataset(sub):
            return sub
    raise FileNotFoundError("Struktur dataset tidak dikenali setelah unduhan.")

DATA_ROOT = resolve_dataset()
print("Dataset root :", DATA_ROOT.resolve())
print("Train images :", len(list((DATA_ROOT / 'train' / 'images').glob('*'))))
print("Val images   :", len(list((DATA_ROOT / 'val' / 'images').glob('*'))))

### 3.1 Pembaca Citra yang Robust

Satu berkas pada dataset berekstensi `.jpg` namun sebenarnya berformat **AVIF**, sehingga
`cv2.imread()` mengembalikan `None`. Karena itu digunakan pembaca dengan fallback ke `Pillow`.
Penanganan kasus seperti ini penting agar pipeline tidak diam-diam kehilangan data.

In [ ]:
def imread_robust(path) -> np.ndarray:
    """Baca citra sebagai BGR; fallback ke Pillow untuk format tak lazim (mis. AVIF)."""
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is not None:
        return img
    from PIL import Image
    with Image.open(path) as im:
        arr = np.array(im.convert("RGB"))
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)


def parse_label_file(path: Path):
    """Baca label YOLO -> list dict dengan koordinat ternormalisasi (x1,y1,x2,y2)."""
    boxes = []
    for line in path.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls = int(parts[0])
        x, y, w, h = map(float, parts[1:5])
        boxes.append({"cls": cls,
                      "x1": x - w / 2, "y1": y - h / 2,
                      "x2": x + w / 2, "y2": y + h / 2})
    return boxes


def build_index(root: Path) -> pd.DataFrame:
    """Bangun tabel: satu baris = satu citra, beserta path & jumlah objek per kelas."""
    rows = []
    for split in ("train", "val"):
        for lbl in sorted((root / split / "labels").glob("*.txt")):
            stem = lbl.stem
            img_path = root / split / "images" / (stem + ".jpg")
            if not img_path.exists():
                # cari ekstensi lain
                hits = list((root / split / "images").glob(stem + ".*"))
                if not hits:
                    continue
                img_path = hits[0]
            boxes = parse_label_file(lbl)
            row = {"split": split, "stem": stem, "image_path": str(img_path),
                   "n_objects": len(boxes), "boxes": boxes}
            for cid, cname in enumerate(CLASS_NAMES):
                row["n_" + cname] = sum(1 for b in boxes if b["cls"] == cid)
            rows.append(row)
    return pd.DataFrame(rows)

df = build_index(DATA_ROOT)
print(f"Total citra terindeks: {len(df)}")
df.head(3)[["split", "stem", "n_objects", "n_with helmet", "n_without helmet", "n_rider", "n_number plate"]]

Total citra terindeks: 124


,split,stem,n_objects,n_with helmet,n_without helmet,n_rider,n_number plate
0,train,new1,3,1,0,1,1
1,train,new10,4,0,2,1,1
2,train,new100,4,0,2,1,1


In [ ]:
# Verifikasi bahwa seluruh citra dapat dibaca
bad = []
for p in df["image_path"]:
    if imread_robust(p) is None:
        bad.append(p)
print("Citra gagal dibaca:", len(bad), bad)

total = {c: int(df["n_" + c].sum()) for c in CLASS_NAMES}
print("\nJumlah bounding box per kelas:")
for k, v in total.items():
    print(f"  {k:16s} : {v}")
print(f"  {'TOTAL':16s} : {sum(total.values())}")